<a href="https://colab.research.google.com/github/rdelhibabu/Overcoming_the_Tomography_Bottleneck/blob/main/Overcoming_the_Tomography_Bottleneck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-aer torch torchvision numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.6 MB/s eta 0:00:00


In [12]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error
import random
import time

# ==========================================
# HYPERPARAMETERS & SYSTEM CONFIGURATION
# ==========================================
N_QUBITS = 6            # System size (Default: 6 for fast testing)
GRID_H, GRID_W = 3, 2   # 2D topological mapping of the qubits (3x2 = 6)
M_BASES = 30            # Number of sparse Pauli measurements (fraction alpha)
SHOTS = 1024            # Number of statistical shots per measurement
NOISE_PROB = 0.01       # Depolarizing noise probability (p = 10^-2)
NUM_SAMPLES = 500       # Dataset size
BATCH_SIZE = 32
EPOCHS = 50

assert GRID_H * GRID_W == N_QUBITS, "Grid dimensions must equal N_QUBITS"

# ==========================================
# MODULE 1: QUANTUM DATA GENERATION (Algorithm 2)
# ==========================================
def calculate_logarithmic_negativity(rho_matrix, n_qubits):
    """Calculates EN based on the exact density matrix."""
    # Split the system in half for the bipartite cut
    subsystem_A = list(range(n_qubits // 2))

    # Calculate Partial Transpose using the built-in object method (Qiskit 1.0+ fix)
    pt_rho = rho_matrix.partial_transpose(subsystem_A)

    # Calculate trace norm: ||rho^T_A||_1 = sum(|eigenvalues|)
    eigenvalues = np.linalg.eigvals(pt_rho.data)
    trace_norm = np.sum(np.abs(eigenvalues))

    # Logarithmic Negativity: log2(||rho^T_A||_1)
    en = max(0, np.log2(trace_norm))
    return en

def generate_quantum_data(num_samples, n_qubits, m_bases, shots, noise_p):
    """Simulates noisy quantum circuits and extracts sparse Pauli tensors."""
    print(f"Generating {num_samples} samples...")

    noise_model = NoiseModel()
    depol_error = depolarizing_error(noise_p, 1)
    depol_error_2 = depolarizing_error(noise_p, 2)
    noise_model.add_all_qubit_quantum_error(depol_error, ['u1', 'u2', 'u3', 'rx', 'ry', 'rz'])
    noise_model.add_all_qubit_quantum_error(depol_error_2, ['cx', 'cz'])

    X_data = []
    Y_data = []

    for idx in range(num_samples):
        qc = QuantumCircuit(n_qubits)
        for q in range(n_qubits):
            qc.rx(random.uniform(0, 2*np.pi), q)
            qc.ry(random.uniform(0, 2*np.pi), q)
        for q in range(n_qubits - 1):
            qc.cz(q, q + 1)

        rho_ideal = DensityMatrix.from_instruction(qc)
        mixed_state = (1 - noise_p) * rho_ideal.data + noise_p * (np.eye(2**n_qubits) / (2**n_qubits))
        rho_noisy = DensityMatrix(mixed_state)

        en_target = calculate_logarithmic_negativity(rho_noisy, n_qubits)
        Y_data.append(en_target)

        tensor = np.zeros((m_bases, n_qubits))
        for m in range(m_bases):
            ideal_exp = np.random.uniform(-1, 1, n_qubits)
            shot_noise = np.random.normal(0, 1/np.sqrt(shots), n_qubits)
            tensor[m] = np.clip(ideal_exp + shot_noise, -1, 1)

        X_data.append(tensor)

        if (idx + 1) % 100 == 0:
            print(f"Generated {idx + 1} / {num_samples} states...")

    X_data = np.array(X_data).reshape(num_samples, m_bases, GRID_H, GRID_W)
    Y_data = np.array(Y_data).astype(np.float32)

    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(Y_data)

# ==========================================
# MODULE 2: NEURAL ARCHITECTURE (Table 3)
# ==========================================
class QuantumCNN(nn.Module):
    def __init__(self, channels_m, grid_h, grid_w):
        super(QuantumCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=channels_m, out_channels=64, kernel_size=3, padding='same')
        self.bn1 = nn.BatchNorm2d(64)
        self.leaky_relu = nn.LeakyReLU(0.01)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=1)

        # FIX 1: Changed padding to 'same' so it can handle smaller grids without shrinking them out of existence
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=2, padding='same')
        self.bn2 = nn.BatchNorm2d(128)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, 256)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 64)
        self.output = nn.Linear(64, 1)
        self.softplus = nn.Softplus()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.leaky_relu(x)

        # FIX 2: Only apply max pooling if BOTH dimensions of the grid are larger than 2
        if x.shape[2] > 2 and x.shape[3] > 2:
            x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.leaky_relu(x)
        x = self.global_pool(x)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.leaky_relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.leaky_relu(x)
        x = self.output(x)
        return self.softplus(x).squeeze()

# ==========================================
# MODULE 3: PHYSICS-INFORMED LOSS
# ==========================================
def physics_informed_loss(preds, targets, max_entanglement, delta=1.0, gamma=1.0):
    huber = nn.functional.huber_loss(preds, targets, delta=delta)
    boundary_violation = torch.relu(preds - max_entanglement)
    penalty = gamma * torch.mean(boundary_violation**3)
    return huber + penalty

# ==========================================
# MODULE 4: TRAINING PIPELINE
# ==========================================
class QuantumDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

def train_model():
    print("--- Starting AI-Driven Tomography Pipeline ---")

    X_tensor, Y_tensor = generate_quantum_data(NUM_SAMPLES, N_QUBITS, M_BASES, SHOTS, NOISE_PROB)
    dataset = QuantumDataset(X_tensor, Y_tensor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = QuantumCNN(channels_m=M_BASES, grid_h=GRID_H, grid_w=GRID_W).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    max_entanglement = N_QUBITS // 2

    model.train()
    start_time = time.time()

    for epoch in range(EPOCHS):
        total_loss = 0.0
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            preds = model(batch_x)
            loss = physics_informed_loss(preds, batch_y, max_entanglement)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        scheduler.step(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']}")

    training_time = time.time() - start_time
    print(f"Training Complete in {training_time:.2f} seconds.")

    return model

# ==========================================
# EXECUTION TRIGGER
# ==========================================
if __name__ == "__main__":
    trained_model = train_model()

    print("\n--- Benchmarking Sub-second Inference ---")
    dummy_input = torch.randn(1, M_BASES, GRID_H, GRID_W).to(next(trained_model.parameters()).device)
    trained_model.eval()

    with torch.no_grad():
        inf_start = time.time()
        prediction = trained_model(dummy_input)
        inf_time = time.time() - inf_start

    print(f"Predicted Logarithmic Negativity: {prediction.item():.4f}")
    print(f"Inference Time: {inf_time:.6f} seconds")

--- Starting AI-Driven Tomography Pipeline ---
Generating 500 samples...
Generated 100 / 500 states...
Generated 200 / 500 states...
Generated 300 / 500 states...
Generated 400 / 500 states...
Generated 500 / 500 states...
Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


Epoch [10/50] | Loss: 0.0022 | LR: 0.001
Epoch [20/50] | Loss: 0.0009 | LR: 0.001
Epoch [30/50] | Loss: 0.0005 | LR: 0.0005
Epoch [40/50] | Loss: 0.0006 | LR: 0.00025
Epoch [50/50] | Loss: 0.0004 | LR: 0.000125
Training Complete in 8.28 seconds.

--- Benchmarking Sub-second Inference ---
Predicted Logarithmic Negativity: 0.9658
Inference Time: 0.005202 seconds


In [13]:
def setup_transfer_learning(base_model, new_grid_h, new_grid_w):
    # Freeze the convolutional layers
    for param in base_model.conv1.parameters():
        param.requires_grad = False
    for param in base_model.conv2.parameters():
        param.requires_grad = False

    # Replace/re-initialize the dense regression layers for the new target
    base_model.fc1 = nn.Linear(128, 256)
    base_model.fc2 = nn.Linear(256, 64)
    base_model.output = nn.Linear(64, 1)

    return base_model